In [1]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
df = pd.read_csv('/content/drive/MyDrive/monkeytype/mt_dataset_pref4feat16.csv')

#df = df[df['x1_participant_F']==1]
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4675 entries, 0 to 4674
Columns: 164 entries, x1_linspace to x4_f15
dtypes: float64(160), int64(4)
memory usage: 5.8 MB


In [28]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


# -----------------------------
# 1) Колонки
# -----------------------------
feat_names = [
    #"linspace",
    "linspace_session_id",
    "linspace_within_session",
    #"brightness",
    #"time_reaction","omission_percent",
    #"symbol_m", "symbol_c", "symbol_s", "symbol_y", "symbol_f", "symbol_j",
    "location_0", "location_1", "location_2", "location_3",
    "participant_J", "participant_F", "participant_Y",
    #"correct_symbol_m", "correct_symbol_c", "correct_symbol_s", "correct_symbol_y", "correct_symbol_f", "correct_symbol_j",
]
latent_dim = 16
feat_names_fs = [f"f{i}" for i in range(latent_dim)]
feat_names.extend(feat_names_fs)

F_DIM = len(feat_names)

pj = feat_names.index("participant_J")
pf = feat_names.index("participant_F")
py = feat_names.index("participant_Y")
participant_idx = (pj, pf, py)

def cols(prefix: str):
    return [f"{prefix}_{f}" for f in feat_names]

x1_cols = cols("x1")
x2_cols = cols("x2")
x3_cols = cols("x3")
x4_cols = cols("x4")


def prepare_arrays(df: pd.DataFrame):
    needed = x1_cols + x2_cols + x3_cols + x4_cols
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    X1 = df[x1_cols].to_numpy(dtype=np.float32)
    X2 = df[x2_cols].to_numpy(dtype=np.float32)
    X3 = df[x3_cols].to_numpy(dtype=np.float32)
    X4 = df[x4_cols].to_numpy(dtype=np.float32)
    return X1, X2, X3, X4


# -----------------------------
# 2) Dataset: возвращает четверку
# -----------------------------
class QuadDataset(Dataset):
    def __init__(self, X1, X2, X3, X4):
        self.X1 = torch.from_numpy(X1).float()
        self.X2 = torch.from_numpy(X2).float()
        self.X3 = torch.from_numpy(X3).float()
        self.X4 = torch.from_numpy(X4).float()

    def __len__(self):
        return self.X1.shape[0]

    def __getitem__(self, idx):
        return self.X1[idx], self.X2[idx], self.X3[idx], self.X4[idx]


# -----------------------------
# 3) Reward model r(x): MLP -> scalar
# -----------------------------
class RewardMLP(nn.Module):
    def __init__(self, in_dim: int, hidden=(64, 32), dropout=0.2):
        super().__init__()
        layers = []
        d = in_dim
        for h in hidden:
            layers += [nn.Linear(d, h), nn.ReLU(), nn.Dropout(dropout)] #nn.ReLU()
            d = h
        layers += [nn.Linear(d, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)  # (B,)

class SwiGLULayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, out_dim)
        self.fc2 = nn.Linear(in_dim, out_dim)

    def forward(self, x):
        return self.fc1(x) * F.silu(self.fc2(x))

class SwiGLUWithParticipantGate(nn.Module):
    def __init__(self, in_dim, out_dim, participant_dim=3, gate_bias=1.5):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, out_dim)
        self.fc2 = nn.Linear(in_dim, out_dim)

        self.p_gate = nn.Linear(participant_dim, out_dim)
        nn.init.zeros_(self.p_gate.weight)
        nn.init.constant_(self.p_gate.bias, gate_bias)

    def forward(self, x, p):
        h = self.fc1(x) * F.silu(self.fc2(x))
        gate = torch.sigmoid(self.p_gate(p))
        return h * gate

class ParticipantGate(nn.Module):
    """
    Обучаемый гейт по participant: g(p) = sigmoid(W p + b), размер (H,)
    p: one-hot (B,3) -> gate: (B,H) -> h' = h * gate
    """
    def __init__(self, hidden_dim: int, init_bias: float = 2.0):
        super().__init__()
        self.linear = nn.Linear(3, hidden_dim)
        # Инициализируем так, чтобы gate был ~0.88 (sigmoid(2)=0.88) => старт почти "без маски"
        nn.init.zeros_(self.linear.weight)
        nn.init.constant_(self.linear.bias, init_bias)

    def forward(self, h: torch.Tensor, participant_onehot: torch.Tensor):
        gate = torch.sigmoid(self.linear(participant_onehot))  # (B,H) в (0,1)
        return h * gate, gate

class RewardMLPLearnableMask(nn.Module):
    def __init__(
        self,
        in_dim: int,
        participant_idx: tuple[int, int, int],  # индексы participant_J/F/Y во входе x
        hidden=(64, 32),
        gate_after_layer=0,      # 0 => после первого hidden слоя
    ):
        super().__init__()
        self.participant_idx = participant_idx
        self.gate_after_layer = gate_after_layer

        self.fcs = nn.ModuleList()
        d = in_dim
        for h in hidden:
            self.fcs.append(nn.Linear(d, h))
            d = h

        self.act = nn.ReLU()

        # gate нужен для слоя hidden[gate_after_layer]
        gate_dim = hidden[gate_after_layer]
        self.gate = ParticipantGate(gate_dim, init_bias=1.5)

        self.head = nn.Linear(d, 1)

    def forward(self, x: torch.Tensor):
        pj, pf, py = self.participant_idx
        p = x[:, [pj, pf, py]]  # (B,3)

        h = x
        last_gate = None

        for i, fc in enumerate(self.fcs):
            h = self.act(fc(h)) #self.act()
            if i == self.gate_after_layer:
                h = self.gate(h, p)

        s = self.head(h).squeeze(-1)

        # вернём gate (для регуляризации/диагностики), но можно и не возвращать
        return s

class RewardMLPLearnableMaskSwiGLU_OneGate(nn.Module):
    def __init__(
        self,
        in_dim,
        participant_idx,
        hidden=(64, 32),
        gate_after_layer=0,
        gate_bias=2.0,
        dropout=0.2,
        return_gate=False,
    ):
        super().__init__()
        self.participant_idx = participant_idx
        self.gate_after_layer = gate_after_layer
        self.return_gate = return_gate

        self.layers = nn.ModuleList()
        self.dropouts = nn.ModuleList()

        d = in_dim
        for h in hidden:
            self.layers.append(SwiGLULayer(d, h))
            self.dropouts.append(nn.Dropout(dropout))
            d = h

        self.gate = ParticipantGate(hidden[gate_after_layer], init_bias=gate_bias)
        self.head = nn.Linear(d, 1)

    def forward(self, x):
        pj, pf, py = self.participant_idx
        p = x[:, [pj, pf, py]]

        h = x
        last_gate = None

        for i, (layer, drop) in enumerate(zip(self.layers, self.dropouts)):
            h = layer(h)

            if i == self.gate_after_layer:
                h, last_gate = self.gate(h, p)
            h = drop(h)

        s = self.head(h).squeeze(-1)
        return s, last_gate if self.return_gate else s

class RewardMLPLearnableMaskSwiGLU(nn.Module):
    """
    Reward MLP для скаляра s=r(x) с SwiGLU + participant-gate в каждом hidden слое.
    """
    def __init__(
        self,
        in_dim: int,
        participant_idx: tuple[int, int, int],
        hidden=(64, 32),
        gate_bias: float = 1.0,
        return_gates: bool = True,
    ):
        super().__init__()
        self.participant_idx = participant_idx
        self.return_gates = return_gates

        self.blocks = nn.ModuleList()
        d = in_dim
        for h in hidden:
            self.blocks.append(SwiGLUWithParticipantGate(d, h, gate_bias=gate_bias))
            d = h

        self.head = nn.Linear(d, 1)

    def forward(self, x: torch.Tensor):
        pj, pf, py = self.participant_idx
        p = x[:, [pj, pf, py]]  # (B,3)

        h = x
        gates = []
        for block in self.blocks:
            h, g = block(h, p)
            if self.return_gates:
                gates.append(g)

        s = self.head(h).squeeze(-1)
        return s, gates if self.return_gates else s

class RewardMLPAllGates(nn.Module):
    """
    MLP reward model, где обучаемый participant-gate стоит ПОСЛЕ КАЖДОГО hidden-слоя.
    """
    def __init__(
        self,
        in_dim: int,
        participant_idx: tuple[int, int, int],
        hidden=(64, 32),
        init_bias=1.0,
        return_gates: bool = False,   # если True — вернет (score, [g1,g2,...])
    ):
        super().__init__()
        self.participant_idx = participant_idx
        self.return_gates = return_gates

        self.fcs = nn.ModuleList()
        self.gates = nn.ModuleList()

        d = in_dim
        for hdim in hidden:
            self.fcs.append(nn.Linear(d, hdim))
            self.gates.append(ParticipantGate(hdim, init_bias=init_bias))
            d = hdim

        self.act = nn.ReLU()
        self.head = nn.Linear(d, 1)

    def forward(self, x: torch.Tensor):
        pj, pf, py = self.participant_idx
        p = x[:, [pj, pf, py]]  # (B,3)

        h = x
        gates_out = []  # для диагностики/регуляризации

        for fc, gate_layer in zip(self.fcs, self.gates):
            h = self.act(fc(h))
            h, g = gate_layer(h, p)
            if self.return_gates:
                gates_out.append(g)

        s = self.head(h).squeeze(-1)
        if self.return_gates:
            return s, gates_out
        return s

class RewardMLP_SwiGLUFirst(nn.Module):
    """
    Первый слой SwiGLU, остальные: Linear -> ReLU -> Dropout.
    Возвращает scalar score s.
    """
    def __init__(
        self,
        in_dim: int,
        hidden=(64, 32),      # hidden[0] = размер после SwiGLU
        dropout=0.2,
    ):
        super().__init__()
        assert len(hidden) >= 1, "hidden must have at least one layer size"

        # 1) первый слой — SwiGLU
        self.swiglu = SwiGLULayer(in_dim, hidden[0])

        # 2) остальные слои — обычные
        blocks = []
        d = hidden[0]
        for h in hidden[1:]:
            blocks += [nn.Linear(d, h), nn.ReLU(), nn.Dropout(dropout)]
            d = h
        self.mlp = nn.Sequential(*blocks) if blocks else nn.Identity()

        # 3) голова
        self.head = nn.Linear(d, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.swiglu(x)        # (B, hidden[0])
        h = self.mlp(h)           # (B, d)
        return self.head(h).squeeze(-1)

class SwiGLUBlock(nn.Module):
    """
    SwiGLU FFN block: x -> swiglu(expanded) -> linear(back to out_dim)
    """
    def __init__(self, in_dim: int, out_dim: int, ff_mult: int = 2):
        super().__init__()
        self.swiglu = SwiGLULayer(in_dim, out_dim * ff_mult)
        self.proj = nn.Linear(out_dim * ff_mult, out_dim)

    def forward(self, x):
        return self.proj(self.swiglu(x))


class RewardMLP_SwiGLUFirst_FF(nn.Module):
    def __init__(self, in_dim: int, hidden=(64, 32), ff_mult=4, dropout=0.2):
        super().__init__()
        assert len(hidden) >= 1

        self.first = SwiGLUBlock(in_dim, hidden[0], ff_mult=ff_mult)

        blocks = []
        d = hidden[0]
        for h in hidden[1:]:
            blocks += [nn.Linear(d, h), nn.ReLU(), nn.Dropout(dropout)]
            d = h
        self.mlp = nn.Sequential(*blocks) if blocks else nn.Identity()
        self.head = nn.Linear(d, 1)

    def forward(self, x):
        h = self.first(x)
        h = self.mlp(h)
        return self.head(h).squeeze(-1)

class GEGLULayer(nn.Module):
    """
    h = (W1 x) * GELU(W2 x)
    """
    def __init__(self, in_dim: int, out_dim: int):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, out_dim)
        self.fc2 = nn.Linear(in_dim, out_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc1(x) * F.gelu(self.fc2(x))

class RewardMLP_GEGLUFirst(nn.Module):
    """
    Первый слой: GEGLU
    Остальные: Linear -> ReLU -> Dropout
    """
    def __init__(
        self,
        in_dim: int,
        hidden=(64, 32),
        dropout=0.2,
    ):
        super().__init__()
        assert len(hidden) >= 1

        # 1) первый слой — GEGLU
        self.geglu = GEGLULayer(in_dim, hidden[0])

        # 2) обычные слои
        blocks = []
        d = hidden[0]
        for h in hidden[1:]:
            blocks += [
                nn.Linear(d, h),
                nn.ReLU(),
                nn.Dropout(dropout),
            ]
            d = h
        self.mlp = nn.Sequential(*blocks) if blocks else nn.Identity()

        # 3) head
        self.head = nn.Linear(d, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.geglu(x)
        h = self.mlp(h)
        return self.head(h).squeeze(-1)

# -----------------------------
# 4) Loss функции
# -----------------------------

def loss_listwise_top1(s1, s2, s3, s4, temperature=1.0, l2_scores=1e-5):
    """
    Listwise: -log softmax winner (x1).
    L = -s1 + logsumexp([s1,s2,s3,s4])
    """
    scores0 = torch.stack([s1, s2, s3, s4], dim=1)
    scores = torch.stack([s1, s2, s3, s4], dim=1) / temperature  # (B,4)
    # target winner index = 0
    loss = F.cross_entropy(scores, torch.zeros(scores.size(0), dtype=torch.long, device=scores.device))
    # small penalty to keep scores bounded
    loss = loss + l2_scores * (scores0**2).mean()
    return loss

def loss_ranknet_pairwise(s1, s2, s3, s4):
    """
    Pairwise RankNet: sum softplus(-(s1-sj)) for j=2..4
    """
    return (F.softplus(-(s1 - s2)) + F.softplus(-(s1 - s3)) + F.softplus(-(s1 - s4))).mean()


# -----------------------------
# 5) Eval метрики
# -----------------------------
@torch.no_grad()
def eval_metrics(model, loader, device, loss_type="listwise", temperature=1.0):
    """
    Оценка на loader для задач top-1 из 4:
      - loss (listwise или pairwise)
      - top1_acc_soft: top-1 с корректной обработкой ties (частичный кредит 1/k)
      - top1_acc_hard: обычный argmax top-1 (может завышать при ties)
      - tie_rate: доля примеров, где максимум не уникален
      - mean_margin: mean(s1 - max(s2,s3,s4))
      - pwin_*: статистики по P(x1 wins) = softmax(scores/T)[0]
    """
    model.eval()

    total_loss = 0.0
    total = 0

    # top-1 метрики
    top1_soft_sum = 0.0
    top1_hard_correct = 0

    # tie diagnostics
    tie_count = 0

    # margin
    margin_sum = 0.0

    # P(win) stats
    p_list = []

    for x1, x2, x3, x4 in loader:
        x1, x2, x3, x4 = x1.to(device), x2.to(device), x3.to(device), x4.to(device)

        s1 = model(x1); s2 = model(x2); s3 = model(x3); s4 = model(x4)
        scores = torch.stack([s1, s2, s3, s4], dim=1)  # (B,4)

        # loss
        if loss_type == "listwise":
            logits = scores / temperature
            loss = F.cross_entropy(
                logits,
                torch.zeros(logits.size(0), dtype=torch.long, device=logits.device)
            )
        elif loss_type == "pairwise":
            loss = (F.softplus(-(s1 - s2)) + F.softplus(-(s1 - s3)) + F.softplus(-(s1 - s4))).mean()
        else:
            raise ValueError("loss_type must be 'listwise' or 'pairwise'")

        bs = scores.size(0)
        total_loss += float(loss.item()) * bs
        total += bs

        # ----- hard top1 (может врать при ties)
        pred = scores.argmax(dim=1)
        top1_hard_correct += int((pred == 0).sum().item())

        # ----- soft top1 (корректно учитывает ties)
        maxv = scores.max(dim=1, keepdim=True).values          # (B,1)
        is_max = (scores == maxv).float()                      # (B,4)
        k = is_max.sum(dim=1)                                  # (B,) сколько максимумов
        # частичный кредит: если x1 среди максимумов, то 1/k
        top1_soft_sum += float((is_max[:, 0] / k).sum().item())

        # tie rate: где максимум не уникален (k>1)
        tie_count += int((k > 1).sum().item())

        # margin: s1 - max(s2,s3,s4)
        best_loser = torch.max(scores[:, 1:], dim=1).values
        margin = (scores[:, 0] - best_loser)
        margin_sum += float(margin.mean().item()) * bs

        # P(win) согласованно с температурой
        pwin = torch.softmax(scores / temperature, dim=1)[:, 0]
        p_list.append(pwin.detach().cpu().numpy())

    p = np.concatenate(p_list) if len(p_list) else np.array([np.nan], dtype=np.float32)

    def p_stats(arr):
        return {
            "mean": float(np.mean(arr)),
            "std":  float(np.std(arr)),
            "q05":  float(np.quantile(arr, 0.05)),
            "q50":  float(np.quantile(arr, 0.50)),
            "q95":  float(np.quantile(arr, 0.95)),
            "min":  float(np.min(arr)),
            "max":  float(np.max(arr)),
        }

    return {
        "loss": total_loss / max(total, 1),
        "top1_acc_soft": top1_soft_sum / max(total, 1),
        "top1_acc_hard": top1_hard_correct / max(total, 1),
        "tie_rate": tie_count / max(total, 1),
        "mean_margin": margin_sum / max(total, 1),
        "pwin": p_stats(p),
    }

@torch.no_grad()
def score_summary(model, loader, device):
    """
    Сводка по абсолютным score s и margin на даталоадере.
    s = r(x) для всех кандидатов
    margin = s1 - max(s2,s3,s4) для каждой четверки
    """
    model.eval()
    all_s = []
    all_margin = []

    for x1, x2, x3, x4 in loader:
        x1, x2, x3, x4 = x1.to(device), x2.to(device), x3.to(device), x4.to(device)

        s1 = model(x1); s2 = model(x2); s3 = model(x3); s4 = model(x4)

        scores = torch.stack([s1, s2, s3, s4], dim=1)  # (B,4)
        all_s.append(scores.detach().cpu().numpy())

        best_loser = torch.max(torch.stack([s2, s3, s4], dim=1), dim=1).values
        margin = (s1 - best_loser)
        all_margin.append(margin.detach().cpu().numpy())

    S = np.concatenate(all_s, axis=0).reshape(-1)      # все s для всех кандидатов
    M = np.concatenate(all_margin, axis=0).reshape(-1) # margin на каждую четверку

    def stats(arr):
        return {
            "mean": float(np.mean(arr)),
            "std":  float(np.std(arr)),
            "q05":  float(np.quantile(arr, 0.05)),
            "q50":  float(np.quantile(arr, 0.50)),
            "q95":  float(np.quantile(arr, 0.95)),
            "min":  float(np.min(arr)),
            "max":  float(np.max(arr)),
        }

    return {"s": stats(S), "margin": stats(M)}


@torch.no_grad()
def winprob_summary(model, loader, device, temperature=1.0):
    """
    Сводка по вероятности выбора x1 (winner) по softmax:
    p = softmax([s1,s2,s3,s4]/T)[0]
    """
    model.eval()
    probs = []

    for x1, x2, x3, x4 in loader:
        x1, x2, x3, x4 = x1.to(device), x2.to(device), x3.to(device), x4.to(device)
        s1 = model(x1); s2 = model(x2); s3 = model(x3); s4 = model(x4);
        scores = torch.stack([s1, s2, s3, s4], dim=1) / temperature
        p = torch.softmax(scores, dim=1)[:, 0]  # (B,)
        probs.append(p.detach().cpu().numpy())

    p = np.concatenate(probs)

    return {
        "mean": float(p.mean()),
        "std":  float(p.std()),
        "q05":  float(np.quantile(p, 0.05)),
        "q50":  float(np.quantile(p, 0.50)),
        "q95":  float(np.quantile(p, 0.95)),
        "min":  float(p.min()),
        "max":  float(p.max()),
    }

def _filter_test_by_participant_onehot(
    df_test: pd.DataFrame,
    participant: str,          # "J" / "F" / "Y"
    *,
    require_consistent=True,   # проверять, что participant одинаков для x1..x4
) -> pd.DataFrame:
    if participant not in ("J", "F", "Y"):
        raise ValueError("participant must be one of: 'J', 'F', 'Y'")

    c1 = f"x1_participant_{participant}"
    c2 = f"x2_participant_{participant}"
    c3 = f"x3_participant_{participant}"
    c4 = f"x4_participant_{participant}"

    missing = [c for c in (c1, c2, c3, c4) if c not in df_test.columns]
    if missing:
        raise ValueError(f"Missing participant one-hot columns: {missing}")

    sub = df_test[df_test[c1] == 1.0].copy()

    if require_consistent and len(sub) > 0:
        # проверяем, что по этой же букве в x2..x4 тоже 1.0
        ok = (sub[c2] == 1.0) & (sub[c3] == 1.0) & (sub[c4] == 1.0)
        # если есть несогласованные строки — выкинем их, чтобы метрика была честной
        sub = sub[ok].copy()

    return sub


@torch.no_grad()
def eval_test_for_participant(
    df_test: pd.DataFrame,
    participant: str,  # "J"/"F"/"Y"
    model,
    scaler,
    device,
    *,
    loss_type="listwise",
    temperature=1.0,
    batch_size=256,
    require_consistent=True,
):
    """
    Считает метрики на тестовой выборке только для строк конкретного participant.

    Возвращает dict:
      - n: сколько строк в подвыборке
      - metrics: результат eval_metrics (или None если n=0)
    """
    sub = _filter_test_by_participant_onehot(
        df_test, participant, require_consistent=require_consistent
    )
    n = len(sub)
    if n == 0:
        return {"participant": participant, "n": 0, "metrics": None}

    # arrays из df
    X1, X2, X3, X4 = prepare_arrays(sub)

    # scaler (fit на train!) применяем одинаково
    def tr(x):
        return scaler.transform(x).astype(np.float32)

    X1, X2, X3, X4 = tr(X1), tr(X2), tr(X3), tr(X4)

    loader = DataLoader(QuadDataset(X1, X2, X3, X4), batch_size=batch_size, shuffle=False)
    metrics = eval_metrics(model, loader, device, loss_type=loss_type, temperature=temperature)

    return {"participant": participant, "n": n, "metrics": metrics}


def eval_test_all_participants(
    df_test: pd.DataFrame,
    model,
    scaler,
    device,
    *,
    participants=("J", "F", "Y"),
    loss_type="listwise",
    temperature=1.0,
    batch_size=256,
    require_consistent=True,
) -> pd.DataFrame:
    """
    Возвращает таблицу метрик по каждому participant.
    """
    rows = []
    for p in participants:
        out = eval_test_for_participant(
            df_test, p, model, scaler, device,
            loss_type=loss_type,
            temperature=temperature,
            batch_size=batch_size,
            require_consistent=require_consistent,
        )
        n = out["n"]
        m = out["metrics"]
        if n == 0 or m is None:
            rows.append({
                "participant": p,
                "n": n,
                "loss": np.nan,
                "top1_soft": np.nan,
                "top1_hard": np.nan,
                "tie_rate": np.nan,
                "mean_margin": np.nan,
                "pwin_mean": np.nan,
                "pwin_q05": np.nan,
                "pwin_q95": np.nan,
            })
            continue

        rows.append({
            "participant": p,
            "n": n,
            "loss": m["loss"],
            "top1_soft": m["top1_acc_soft"],
            "top1_hard": m["top1_acc_hard"],
            "tie_rate": m["tie_rate"],
            "mean_margin": m["mean_margin"],
            "pwin_mean": m["pwin"]["mean"],
            "pwin_q05": m["pwin"]["q05"],
            "pwin_q95": m["pwin"]["q95"],
        })

    return pd.DataFrame(rows).sort_values("n", ascending=False)

# -----------------------------
# 6) Train
# -----------------------------
def train_ranknet_quads(
    df: pd.DataFrame,
    *,
    loss_type="listwise",
    temperature=1.0,

    test_size=0.2,      # доля TEST от всего df
    val_size=0.25,       # доля VAL от оставшегося после test

    random_state=0,
    batch_size=64,
    epochs=60,
    lr=1e-3,
    weight_decay=1e-4,
    hidden=(64, 32),
    dropout=0.1,
    device=None,
    gate_bias = 2.5,
    monitor="val_top1_acc", #val_loss or val_top1_acc
    min_delta=0.0,
    restore_best=True,
):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    # --- Split по индексам, чтобы легко вернуть df_test ---
    idx = np.arange(len(df))

    idx_tmp, idx_te = train_test_split(
        idx, test_size=test_size, random_state=random_state, shuffle=True
    )
    idx_tr, idx_va = train_test_split(
        idx_tmp, test_size=val_size, random_state=random_state, shuffle=True
    )

    df_tr = df.iloc[idx_tr].reset_index(drop=True)
    df_va = df.iloc[idx_va].reset_index(drop=True)
    df_te = df.iloc[idx_te].reset_index(drop=True)

    #df_tr = df_tr[df_tr['x1_participant_Y']==1]
    #df_va = df_va[df_va['x1_participant_Y']==1]
    #df_te = df_te[df_te['x1_participant_Y']==1]

    # --- arrays ---
    X1_tr, X2_tr, X3_tr, X4_tr = prepare_arrays(df_tr)
    X1_va, X2_va, X3_va, X4_va = prepare_arrays(df_va)
    X1_te, X2_te, X3_te, X4_te = prepare_arrays(df_te)

    # --- scaler fit только на TRAIN кандидатах ---
    scaler = StandardScaler()
    scaler.fit(np.vstack([X1_tr, X2_tr, X3_tr, X4_tr]))

    def tr(x): return scaler.transform(x).astype(np.float32)

    X1_tr, X2_tr, X3_tr, X4_tr = tr(X1_tr), tr(X2_tr), tr(X3_tr), tr(X4_tr)
    X1_va, X2_va, X3_va, X4_va = tr(X1_va), tr(X2_va), tr(X3_va), tr(X4_va)
    X1_te, X2_te, X3_te, X4_te = tr(X1_te), tr(X2_te), tr(X3_te), tr(X4_te)

    train_loader = DataLoader(QuadDataset(X1_tr, X2_tr, X3_tr, X4_tr),
                              batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(QuadDataset(X1_va, X2_va, X3_va, X4_va),
                              batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(QuadDataset(X1_te, X2_te, X3_te, X4_te),
                              batch_size=batch_size, shuffle=False)

    model = RewardMLP(in_dim=F_DIM, hidden=hidden, dropout=dropout).to(device)

    #model = RewardMLPLearnableMask(in_dim=F_DIM, participant_idx=participant_idx, hidden=hidden, gate_after_layer=0).to(device)

    #model = RewardMLPAllGates(in_dim=F_DIM, participant_idx=participant_idx, hidden=hidden, init_bias=gate_bias, return_gates=False).to(device)

    #model = RewardMLPLearnableMaskSwiGLU_OneGate(in_dim=F_DIM, participant_idx=participant_idx, hidden=hidden, gate_after_layer=0, gate_bias=gate_bias, dropout=dropout).to(device)

    #model = RewardMLP_SwiGLUFirst(in_dim=F_DIM, hidden=hidden, dropout=dropout).to(device)

    #model = RewardMLP_SwiGLUFirst_FF(in_dim=F_DIM, hidden=hidden, dropout=dropout).to(device)

    #model = RewardMLP_GEGLUFirst(in_dim=F_DIM, hidden=hidden, dropout=dropout).to(device)

    # AdamW предпочтительнее, но оставлю как у тебя Adam, если хочешь:
    optim = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optim, T_max=epochs, eta_min=1e-6
    )

    # --- best tracking ---
    if monitor == "val_loss":
        mode = "min"
        best_score = float("inf")
    elif monitor == "val_top1_acc":
        mode = "max"
        best_score = -float("inf")
    else:
        raise ValueError("monitor must be 'val_loss' or 'val_top1_acc'")

    best_epoch = 0
    best_state = None

    for epoch in range(1, epochs + 1):
        model.train()
        for x1, x2, x3, x4 in train_loader:
            x1, x2, x3, x4 = x1.to(device), x2.to(device), x3.to(device), x4.to(device)

            s1 = model(x1); s2 = model(x2); s3 = model(x3); s4 = model(x4)

            if loss_type == "listwise":
                loss = loss_listwise_top1(s1, s2, s3, s4, temperature=temperature)
            else:
                loss = loss_ranknet_pairwise(s1, s2, s3, s4)

            optim.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optim.step()

        scheduler.step()

        trm = eval_metrics(model, train_loader, device, loss_type=loss_type, temperature=temperature)
        vam = eval_metrics(model, val_loader, device, loss_type=loss_type, temperature=temperature)

        print(
            f"epoch {epoch:03d} | "
            f"train loss {trm['loss']:.4f} top1_soft {trm['top1_acc_soft']:.3f} "
            f"tie {trm['tie_rate']:.3f} margin {trm['mean_margin']:.3f} | "
            f"val loss {vam['loss']:.4f} top1_soft {vam['top1_acc_soft']:.3f} "
            f"tie {vam['tie_rate']:.3f} margin {vam['mean_margin']:.3f} | "
            f"val Pwin mean {vam['pwin']['mean']:.3f} q05 {vam['pwin']['q05']:.3f} q95 {vam['pwin']['q95']:.3f}"
        )

        report = eval_test_all_participants(df_va, model, scaler, device, temperature=temperature, loss_type="listwise")
        print(report)

        current = vam["loss"] if monitor == "val_loss" else vam["top1_acc_soft"]
        improved = (current < best_score - min_delta) if mode == "min" else (current > best_score + min_delta)

        if improved:
            best_score = current
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if restore_best and best_state is not None:
        model.load_state_dict(best_state)
        print(f"Restored best model from epoch {best_epoch} with {monitor}={best_score:.4f}")

    # --- финальная оценка на test ---
    test_metrics = eval_metrics(model, test_loader, device, loss_type=loss_type, temperature=temperature)
    print(
        f"[TEST] loss {test_metrics['loss']:.4f} top1_soft {test_metrics['top1_acc_soft']:.3f} "
        f"tie {test_metrics['tie_rate']:.3f} margin {test_metrics['mean_margin']:.3f} | "
        f"Pwin mean {test_metrics['pwin']['mean']:.3f} q05 {test_metrics['pwin']['q05']:.3f} q95 {test_metrics['pwin']['q95']:.3f}"
    )

    info = {
        "best_epoch": best_epoch,
        "best_score": best_score,
        "monitor": monitor,
        "test_metrics": test_metrics,
        "sizes": {"train": len(df_tr), "val": len(df_va), "test": len(df_te)},
        "indices": {"train": idx_tr, "val": idx_va, "test": idx_te},
    }

    splits = {"df_train": df_tr, "df_val": df_va, "df_test": df_te}
    return model, scaler, device, info, splits

In [34]:
temp = 3.9
model, scaler, device, monitor_dict, splits = train_ranknet_quads(
    df,
    loss_type="listwise",
    temperature=temp,
    epochs=400,
    lr=2e-4,
    weight_decay=1e-2,
    hidden=(256, 192),
    dropout=0.5,
    #gate_bias = 2.9,
    batch_size=256,
)

epoch 001 | train loss 1.3693 top1_soft 0.476 tie 0.000 margin -0.022 | val loss 1.3688 top1_soft 0.487 tie 0.000 margin -0.019 | val Pwin mean 0.254 q05 0.243 q95 0.263
  participant    n      loss  top1_soft  top1_hard  tie_rate  mean_margin  \
0           J  358  1.371018   0.466480   0.466480       0.0    -0.024698   
2           Y  307  1.363635   0.566775   0.566775       0.0     0.010285   
1           F  270  1.371669   0.422222   0.422222       0.0    -0.046173   

   pwin_mean  pwin_q05  pwin_q95  
0   0.253928  0.241383  0.262332  
2   0.255771  0.246499  0.262481  
1   0.253787  0.241239  0.263982  
epoch 002 | train loss 1.3521 top1_soft 0.505 tie 0.000 margin -0.027 | val loss 1.3523 top1_soft 0.524 tie 0.000 margin -0.027 | val Pwin mean 0.259 q05 0.237 q95 0.273
  participant    n      loss  top1_soft  top1_hard  tie_rate  mean_margin  \
0           J  358  1.355622   0.505587   0.505587       0.0    -0.034002   
2           Y  307  1.342106   0.596091   0.596091       

In [32]:
df_test = splits["df_test"]

report = eval_test_all_participants(
    df_test, model, scaler, device,
    temperature=temp, loss_type="listwise"
)
print(report)

  participant    n      loss  top1_soft  top1_hard  tie_rate  mean_margin  \
0           J  347  1.185506   0.510086   0.510086       0.0    -0.665368   
2           Y  305  0.854538   0.596721   0.596721       0.0     0.892422   
1           F  283  1.231804   0.420495   0.420495       0.0    -0.915913   

   pwin_mean  pwin_q05  pwin_q95  
0   0.375094  0.069817  0.661517  
2   0.476778  0.133170  0.760863  
1   0.343574  0.091274  0.610587  


In [ ]:
import os
import json
import pandas as pd
from datetime import datetime


def save_splits_csv(
    *,
    df_train: pd.DataFrame,
    df_val: pd.DataFrame,
    df_test: pd.DataFrame,
    out_dir: str,
    seed: int = None,
    note: str = "",
):
    """
    Сохраняет train / val / test в CSV + metadata.json
    """
    os.makedirs(out_dir, exist_ok=True)

    # --- сами данные ---
    df_train.to_csv(f"{out_dir}/train.csv", index=False)
    df_val.to_csv(f"{out_dir}/val.csv", index=False)
    df_test.to_csv(f"{out_dir}/test.csv", index=False)

    # --- метаданные ---
    meta = {
        "created_at": datetime.utcnow().isoformat(),
        "n_train": len(df_train),
        "n_val": len(df_val),
        "n_test": len(df_test),
        "seed": seed,
        "note": note,
        "columns": list(df_train.columns),
    }

    with open(f"{out_dir}/metadata.json", "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2, ensure_ascii=False)

    print(
        f"Saved splits to '{out_dir}': "
        f"train={len(df_train)}, val={len(df_val)}, test={len(df_test)}"
    )

def load_splits_csv(out_dir: str):
    """
    Загружает train / val / test + metadata.json
    """
    df_train = pd.read_csv(f"{out_dir}/train.csv")
    df_val   = pd.read_csv(f"{out_dir}/val.csv")
    df_test  = pd.read_csv(f"{out_dir}/test.csv")

    meta = None
    meta_path = f"{out_dir}/metadata.json"
    if os.path.exists(meta_path):
        with open(meta_path, "r", encoding="utf-8") as f:
            meta = json.load(f)

    return {
        "df_train": df_train,
        "df_val": df_val,
        "df_test": df_test,
        "metadata": meta,
    }

In [ ]:
save_splits_csv(
    df_train=splits["df_train"],
    df_val=splits["df_val"],
    df_test=splits["df_test"],
    out_dir="/content/drive/MyDrive/monkeytype/splits_v1",
    seed=0,
    note="RankNet quads baseline split"
)